# Compare Different Models on Real-World Datasets

Evaluate all registered models on OpenML benchmark datasets, with per-model cache reuse and W&B logging.

In [ ]:
from pathlib import Path

from pfns.run_evaluation_cli import (
    compute_per_dataset_stats,
    print_results_summary,
    run_tabpfn_evaluation,
    summarize_results,
)
from notebook_utils import (
    single_model_hash,
)
from pfns.utils import get_default_device
from pfns.experiments.model_benchmarks.io import (
    download_results_bundle_from_wandb,
    load_dataframe_bundle,
    make_bundle_path,
    make_model_artifact_name,
    run_metadata_matches,
    sanitize_wandb_artifact_component,
    save_dataframe_bundle,
    upload_results_bundle_to_wandb,
)
from pfns.experiments.model_benchmarks.model_registry import (
    MODEL_FAMILIES,
    get_all_models,
    get_models_from_families,
    get_models_from_names,
)

REAL_DATASET_EXPERIMENT = {
    "name": "real_world_openml_comparison",
    "benchmark": "opencc",
    "max_samples": 1000,
    "max_features": 20,
    "max_classes": 10,
    "n_splits": 5,
    "batch_size_inference": 32,
    "n_ensemble_configurations": 32,
    "preprocess_transforms": ["none", "power", "robust"],
    "sample_order_permutation": True,
    "fla_cache_chunk_size": None,
}

WANDB = {
    "enabled": True,
    "overwrite": False,
    "artifact_name_real_eval": "real_eval_results",
    "entity": "icl_arch",
    "artifact_project": "real_world_eval_artifacts",
    "scores_project": "real_world_eval_scores",
    "mode": "online"
}

EXPECTED_REAL_METADATA_KEYS = (
    "benchmark",
    "max_samples",
    "max_features",
    "max_classes",
    "n_splits",
    "device",
    "batch_size_inference",
    "n_ensemble_configurations",
    "preprocess_transforms",
    "sample_order_permutation",
    "fla_cache_chunk_size",
)

OUTPUT_ROOT = Path.cwd().resolve() / "exp_outputs" / "real_world_eval"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Results are stored in: {OUTPUT_ROOT}")
print(f"Available model families: {list(MODEL_FAMILIES)}")

# Model selection
# models_to_compare = get_models_from_families(["transformer", "gla"])
# models_to_compare = get_models_from_names(["Softmax_Transformer", "GLA_Cached"])
models_to_compare = get_all_models()

device = get_default_device()
print(f"Using device: {device}")

expected_real_metadata = {
    "benchmark": REAL_DATASET_EXPERIMENT["benchmark"],
    "max_samples": REAL_DATASET_EXPERIMENT["max_samples"],
    "max_features": REAL_DATASET_EXPERIMENT["max_features"],
    "max_classes": REAL_DATASET_EXPERIMENT["max_classes"],
    "n_splits": REAL_DATASET_EXPERIMENT["n_splits"],
    "device": device,
    "batch_size_inference": REAL_DATASET_EXPERIMENT["batch_size_inference"],
    "n_ensemble_configurations": REAL_DATASET_EXPERIMENT["n_ensemble_configurations"],
    "preprocess_transforms": list(REAL_DATASET_EXPERIMENT["preprocess_transforms"]),
    "sample_order_permutation": bool(REAL_DATASET_EXPERIMENT["sample_order_permutation"]),
    "fla_cache_chunk_size": REAL_DATASET_EXPERIMENT["fla_cache_chunk_size"],
}


In [ ]:
required_files = ["metadata.json", "results.csv", "summary.csv", "per_dataset.csv"]

if WANDB["enabled"] and WANDB["overwrite"]:
    print("WANDB overwrite=True: skipping per-model download and forcing reruns.")

for model_name, model_config in models_to_compare.items():
    model_hash = single_model_hash(
        model_name=model_name,
        model_config=model_config,
        experiment_payload=REAL_DATASET_EXPERIMENT,
    )
    model_artifact_name = make_model_artifact_name(
        base_artifact_name=WANDB["artifact_name_real_eval"],
        model_name=model_name,
        model_hash=model_hash,
    )

    if WANDB["enabled"] and not WANDB["overwrite"]:
        cached_bundle_path = download_results_bundle_from_wandb(
            artifact_name=model_artifact_name,
            download_root=OUTPUT_ROOT / "wandb_model_cache" / "real_world",
            required_files=required_files,
            entity=WANDB["entity"],
            project=WANDB["artifact_project"],
        )
        if cached_bundle_path is not None:
            cached_bundle = load_dataframe_bundle(
                cached_bundle_path,
                expected_keys=("results", "summary", "per_dataset"),
            )
            cached_results = cached_bundle["dataframes"]["results"]
            has_model = model_name in set(cached_results.get("model", []))
            metadata_ok = run_metadata_matches(
                cached_bundle["metadata"],
                expected=expected_real_metadata,
                keys=EXPECTED_REAL_METADATA_KEYS,
            )
            if has_model and metadata_ok:
                print(f"Reused cached real-world W&B result for {model_name}: {cached_bundle_path}")
                continue

    print(f"Running real-world benchmark for model: {model_name}")
    wandb_run_id = model_config.get("wandb_run_id")
    base_path = model_config.get("base_path")
    if base_path is None and wandb_run_id is None:
        base_path = "."

    results = run_tabpfn_evaluation(
        base_path=base_path,
        checkpoint_name=model_config.get("checkpoint_name", "checkpoint.pt"),
        wandb_run_id=wandb_run_id,
        device=device,
        benchmark=REAL_DATASET_EXPERIMENT["benchmark"],
        max_samples=REAL_DATASET_EXPERIMENT["max_samples"],
        max_features=REAL_DATASET_EXPERIMENT["max_features"],
        max_classes=REAL_DATASET_EXPERIMENT["max_classes"],
        n_splits=REAL_DATASET_EXPERIMENT["n_splits"],
        only_tabpfn=True,
        batch_size_inference=REAL_DATASET_EXPERIMENT["batch_size_inference"],
        n_ensemble_configurations=REAL_DATASET_EXPERIMENT["n_ensemble_configurations"],
        preprocess_transforms=REAL_DATASET_EXPERIMENT["preprocess_transforms"],
        sample_order_permutation=REAL_DATASET_EXPERIMENT["sample_order_permutation"],
        fla_cache_chunk_size=REAL_DATASET_EXPERIMENT["fla_cache_chunk_size"],
        verbose=False,
    )

    if not results.empty:
        results["model"] = model_name
    summary = summarize_results(results)
    per_dataset = compute_per_dataset_stats(results)

    model_bundle_path = make_bundle_path(
        OUTPUT_ROOT / "real_world",
        f"{REAL_DATASET_EXPERIMENT['name']}_{sanitize_wandb_artifact_component(model_name)}",
    )
    save_dataframe_bundle(
        dataframes={
            "results": results,
            "summary": summary.reset_index() if summary is not None else None,
            "per_dataset": per_dataset,
        },
        bundle_dir=model_bundle_path,
        experiment=REAL_DATASET_EXPERIMENT,
        run_metadata=expected_real_metadata,
    )
    print(f"Saved real-world bundle for {model_name}: {model_bundle_path}")

    if WANDB["enabled"]:
        artifact_ref = upload_results_bundle_to_wandb(
            model_bundle_path,
            artifact_name=model_artifact_name,
            run_name=f"{REAL_DATASET_EXPERIMENT['name']}_{sanitize_wandb_artifact_component(model_name)}_{model_hash}_artifact",
            metadata={
                "experiment": REAL_DATASET_EXPERIMENT,
                "model_name": model_name,
                "model_config": model_config,
                "model_hash": model_hash,
                "run_metadata": expected_real_metadata,
            },
            entity=WANDB["entity"],
            project=WANDB["artifact_project"],
            job_type="real_world_bundle_upload",
        )
        print(f"Uploaded real-world artifact for {model_name}: {artifact_ref}")

print("\nReal-world evaluation completed.")
